In [10]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import ExtraTreesRegressor


In [11]:
df = pd.read_csv("total_data_monthly.csv", encoding="utf-8-sig").dropna()
df.head()

,지역명,연도,인구밀도,인구밀도_평균,인구밀도_표준편차,인구밀도_,경사도_평균,경사도_최대,이재민 인,건물피해액,월,월강수량
0,강원특별자치도 강릉시,2012,209.32,-3726.751925,-0.599519,0.006620,17.641871,62.290401,23.0,7800.0,6.0,84.0
1,강원특별자치도 강릉시,2012,209.32,-3726.751925,-0.599519,0.006620,17.641871,62.290401,23.0,7800.0,7.0,1049.8
2,강원특별자치도 강릉시,2012,209.32,-3726.751925,-0.599519,0.006620,17.641871,62.290401,23.0,7800.0,8.0,1001.8
3,강원특별자치도 강릉시,2012,209.32,-3726.751925,-0.599519,0.006620,17.641871,62.290401,23.0,7800.0,9.0,877.5
4,강원특별자치도 강릉시,2013,208.39,-3727.681925,-0.599669,0.006588,17.641871,62.290401,0.0,0.0,6.0,114.6


In [12]:
# 지역-연도별 연 강수량 합
annual_rain = (
    df.groupby(["지역명", "연도"])["월강수량"]
    .transform("sum")
)

# 월별 강수량 가중치
df["rain_weight"] = df["월강수량"] / annual_rain

# 월별 이재민 수 생성
df["월별_이재민"] = df["이재민 인"] * df["rain_weight"]

# 월별 건물피해액 생성
df["월별_건물피해액"] = df["건물피해액"] * df["rain_weight"]

In [13]:
# 로그 변환
df["인구밀도_log"] = np.log1p(df["인구밀도"])
df["월강수량_log"] = np.log1p(df["월강수량"])
df["월별_건물피해액_log"] = np.log1p(df["월별_건물피해액"])
df['월별_이재민_log'] = np.log1p(df['월별_이재민'])

In [14]:
feature_cols = [
    "인구밀도_log",
    "경사도_평균",
    "월강수량_log",
    "월별_건물피해액_log",
]

X = df[feature_cols]
y = df["월별_이재민_log"]

In [15]:
df[feature_cols].head()

,인구밀도_log,경사도_평균,월강수량_log,월별_건물피해액_log
0,5.348630,17.641871,4.442651,5.386559
1,5.348630,17.641871,6.957307,7.907877
2,5.348630,17.641871,6.910551,7.861093
3,5.348630,17.641871,6.778216,7.728671
4,5.344199,17.641871,4.750136,0.000000


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [17]:
model = ExtraTreesRegressor(
    n_estimators=600,
    max_depth=None,        
    min_samples_leaf=1,    
    min_samples_split=2,
    max_features=0.6,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",600
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.6
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the 

In [18]:
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R² score: {r2:.3f}")
print(f"MAE: {mae:.2f} 명")

R² score: 0.753
MAE: 0.27 명


R² = 0.753
입력 변수들이 월별 이재민 수 변동의 약 75%를 설명

MAE = 0.27명
평균적으로 0.3명 이내 오차


In [19]:
importances = pd.Series(
    model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print(importances)

월별_건물피해액_log    0.661342
월강수량_log        0.126822
인구밀도_log        0.121650
경사도_평균          0.090185
dtype: float64


In [ ]:
X_pred = df[feature_cols].copy()
pred_log = model.predict(X_pred)

df["예측_월별_이재민_log"] = pred_log

# 명 단위
df["예측_월별_이재민_명"] = np.expm1(df["예측_월별_이재민_log"]).clip(lower=0)

# 예측값 큰 순으로 확인
df_sorted = df.sort_values("예측_월별_이재민_명", ascending=False)
print("예측 규모 Top 20")
display(df_sorted[["지역명","연도","예측_월별_이재민_명","예측_월별_이재민_log"]].head(20))


✅ 발생(1)로 예측된 중 예측 규모 Top 20


,지역명,연도,예측_월별_이재민_명,예측_월별_이재민_log
3099,부산광역시 동래구,2014,1313.552542,7.181252
2919,부산광역시 기장군,2014,957.465201,6.865333
2395,경상북도 영덕군,2018,776.848000,6.656531
2394,경상북도 영덕군,2018,714.925333,6.573576
4705,전라남도 구례군,2020,614.068394,6.421733
4633,전라남도 곡성군,2020,534.839501,6.283835
4357,울산광역시 울주군,2014,514.243051,6.244639
4704,전라남도 구례군,2020,500.179593,6.216965
4776,전라남도 담양군,2020,482.835197,6.181744
2397,경상북도 영덕군,2019,480.002633,6.175873


모델링 된 결과 저장

In [29]:
df.to_csv('예측모델링.csv',encoding='cp949')

In [51]:
raw_data = pd.read_csv("예측모델링.csv",encoding='cp949')


raw_data = raw_data.copy()

# 1) 연도/월 숫자화
raw_data["연도"] = pd.to_numeric(raw_data["연도"], errors="coerce")
raw_data["월"]   = pd.to_numeric(raw_data["월"], errors="coerce")

# 2) 월 인덱스 만들기 (연도-월을 하나의 시간축으로)
raw_data["ym"] = raw_data["연도"] * 12 + raw_data["월"]

# 3) 최신성 점수 (0~1)
ym_min = raw_data["ym"].min()
ym_max = raw_data["ym"].max()
raw_data["recency_raw"] = (raw_data["ym"] - ym_min) / (ym_max - ym_min + 1e-9)


In [ ]:
raw_data.head(5)

- 1차 모델링 결과 동일 지역이 연도 차이로 인해 여러번 나오는 것을 확인함.
- 위 결과의 경우 어떤 지역이 더 위험한 지 구별하기 어렵기 때문에 위험도 점수를 구해 선별하고자 함.
1. 지역별 집계 생성
2. 지역별 최신성 / 빈도수 / 규모를 기준으로 위험 점수를 계산함
3. 최종 점수를 더해 위험 지역 상위 20개 추출

In [ ]:
# =========================================
# 이재민 수가 적어도 1명 이상으로 예측된 데이터들만 고려
# 0 or 0.n값은 사실상 0명이기 때문에 위험 가중치 선정에 반영하지 말아야 한다고 판단함.
# 이재민이 1명 이상인 데이터 수 : 1371개
# =========================================
thr = 1
raw_data["pred_pos"] = (raw_data["예측_월별_이재민_명"] >= thr).astype(int)

# =========================================
# 2) 지역별 집계: 빈도/규모/최신성
# =========================================
grouped = raw_data.groupby("지역명", dropna=False)

region_scores = grouped.agg(
    n=("pred_pos", "size"), # 지역별로 이재민 수 1명 이상인 값들의 수
    freq=("pred_pos", "mean"), # 빈도 -> 얼마나 자주 발생했는지 
    sev_mean=("예측_월별_이재민_명", lambda s: s[s >= thr].mean() if (s >= thr).any() else 0.0), # 지역별 이재민 수 평균
    sev_sum=("예측_월별_이재민_명", "sum"), # 지역별 이재민 수 누적합
    recency=("recency_raw",
             lambda s: (s * raw_data.loc[s.index, "pred_pos"]).sum() /
                       (raw_data.loc[s.index, "pred_pos"].sum() + 1e-9)) # 최신성
).reset_index()

# 숫자형 변환
for c in ["freq", "sev_mean", "sev_sum", "recency"]:
    region_scores[c] = pd.to_numeric(region_scores[c], errors="coerce").fillna(0)

# =========================================
# 3) 0~1 정규화 (Min-Max) + 규모는 log1p 완화
# =========================================
region_scores["freq_n"] = (
    (region_scores["freq"] - region_scores["freq"].min()) /
    (region_scores["freq"].max() - region_scores["freq"].min() + 1e-9)
)

# 월별 이재민 평균 log 변환 + MinMax 스케일링
sev_base = np.log1p(region_scores["sev_mean"].clip(lower=0))
region_scores["sev_n"] = (sev_base - sev_base.min()) / (sev_base.max() - sev_base.min() + 1e-9)

# 최신성 MinMax 스케일링
region_scores["rec_n"] = (
    (region_scores["recency"] - region_scores["recency"].min()) /
    (region_scores["recency"].max() - region_scores["recency"].min() + 1e-9)
)

# =========================================
# 4) 최종 위험점수(가중합) -> 임의로 가중치를 선정함 / 이재민 수와 최근에 많이 발생한 지역에 더 큰 가중치를 줌
# =========================================
w_freq, w_sev, w_rec = 0.2, 0.5, 0.3
region_scores["risk_score"] = (
    w_freq * region_scores["freq_n"] +
    w_sev  * region_scores["sev_n"] +
    w_rec  * region_scores["rec_n"]
)

# =========================================
# 5) 최종 위험지역 Top N
# =========================================
TOP_N = 20
top_risk_regions = region_scores.sort_values("risk_score", ascending=False).head(TOP_N)

print(f"✅ 최종 위험지역 Top {TOP_N} (빈도+규모+최신성)")
display(top_risk_regions[["지역명", "risk_score", "freq", "sev_mean", "recency", "n"]])



✅ 최종 위험지역 Top 20 (빈도+규모+최신성)


,지역명,risk_score,freq,sev_mean,recency,n
65,경상북도 영덕군,0.838369,0.361111,304.951542,0.798757,36
71,경상북도 울진군,0.742120,0.388889,133.965058,0.692641,36
109,울산광역시 울주군,0.730709,0.527778,83.821259,0.666667,36
2,강원특별자치도 삼척시,0.712798,0.500000,76.894533,0.656566,36
145,충청남도 아산시,0.709237,0.250000,76.752598,0.865320,36
136,제주특별자치도 제주시,0.707006,0.555556,68.555218,0.621212,36
63,경상북도 성주군,0.699090,0.444444,79.282116,0.651515,36
156,충청북도 제천시,0.691195,0.138889,112.726510,0.793939,36
116,전라남도 곡성군,0.689460,0.194444,202.620926,0.571429,36
82,부산광역시 동래구,0.684793,0.333333,121.284705,0.580808,36


In [ ]:
#n = 관측된 월 수

region_scores['n'].unique()

array([ 36,  72,  35,  28, 144, 180,  34])

In [50]:
len(raw_data[raw_data['pred_pos']==1])

1371